In [1]:
import requests
import pandas as pd
import json
import time
from config import *
from datetime import datetime
from zoneinfo import ZoneInfo

API URL

In [2]:
italki = "https://api.italki.com/api/v2/teachers"
language = 'french'

Define the next few functions

In [3]:
def get_data(page, min_price, max_price):
    headers_format = {
        'Accept': 'application/json, text/plain, */*',
        'Accept-Encoding': 'gzip, deflate, br, zstd',
        'Accept-Language': 'en-GB,en-US;q=0.9,en;q=0.8,zh-CN;q=0.7,zh;q=0.6',
        'Content-Length': '130',
        'Content-Type': 'application/json',
        'Origin': 'https://www.italki.com',
        'Priority': 'u=1, i',
        'Referer': 'https://www.italki.com/',
        'Sec-Ch-Ua': '"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand";v="99"',
        'Sec-Ch-Ua-Platform': '?0',
        'Sec-Ch-Ua-Platform': "Windows",
        'Sec-Fetch-Dest': '',
        'Sec-Fetch-Mode': 'cors',
        'Sec-Fetch-Site': 'same-site',
        'traceparent': traceparent,
        'User-Agent': personal_user_agent,
        'x-browser-key': personal_browser_key, # Found in the actual POST request
        'x-device': '10',                       # Common default, but check yours
        'x-locale': 'en',
        'x-token': personal_x_token    
    }

    payload = {
        "teach_language": {
            "language": language,
            "max_price": max_price,
            "min_price": min_price,
        },
        "page": page,
        "page_size": 20,
        "user_timezone": "Asia/Singapore"
    }

    response = requests.post(url = italki, headers = headers_format, json=payload)
    time.sleep(2)
    dictionary = response.json()
    return dictionary

In [4]:
def join_data(new_data, filename):
    with open(filename, 'r') as file:
        existing = json.load(file)
    
    existing_tuples = {tuple(sorted(d.items())) for d in existing}
    new_no_duplicates = []
    seen = set()
    for d in new_data:
        d_tuple = tuple(sorted(d.items()))

        if d_tuple not in seen:
            new_no_duplicates.append(d)
            seen.add(d_tuple)

    new_to_join = [
        d for d in new_no_duplicates
        if tuple(sorted(d.items())) not in existing_tuples
    ]

    combined = existing + new_to_join
    with open(filename, 'w') as file:
        json.dump(combined, file, indent=4)
    return len(new_to_join)

In [5]:
def override_data(new_data, filename):
    with open(filename, 'w') as file:
        json.dump(new_data, file, indent=4)

Appending new information to old JSONs

In [6]:
def user_course_info(teachers):
    user_course_info_list = []
    for teacher in teachers:
        user_course_info = teacher['user_info'] | teacher['course_info']
        del user_course_info['avatar_file_name']
        del user_course_info['is_online']
        del user_course_info['last_login_time']
        del user_course_info['trial_description']
        user_course_info_list.append(user_course_info)
    return user_course_info_list

    # user_course_info
    # df = pd.DataFrame(user_course_info_list)


In [7]:
def teacher_stats(teachers):
    teacher_stats_list = []
    for teacher in teachers:
        test_stat = {}
        test_stat['user_id'] = teacher['user_info']['user_id']
        test_stat['finished_session'] = teacher['teacher_statistics']['finished_session']
        test_stat['response_rate'] = teacher['teacher_statistics']['response_rate']
        test_stat['attendance_rate'] = teacher['teacher_statistics']['attendance_rate']
        teacher_stats_list.append(test_stat)
    return teacher_stats_list

In [8]:
def pro_course_prices(teachers):
    master_price_list = []
    pro_course_detail_list = []
    for teacher in teachers:
        for course in teacher['pro_course_detail']:
            for prices in course['price_list']:
                master_price_list.append(prices)
            course_no_price = course.copy()
            del course_no_price['price_list']
            del course_no_price['description']
            del course_no_price['level_lower_limit']
            del course_no_price['level_up_limit']
            del course_no_price['course_category']
            del course_no_price['course_tags']
            del course_no_price['create_time']
            pro_course_detail_list.append(course_no_price)
    return master_price_list, pro_course_detail_list


In [9]:
with open('also_speaks_reference.json', 'r') as file:
    also_speak_index_list = json.load(file)
def check_language(language):
    if language in also_speak_index_list:
        return also_speak_index_list.index(language)
    else:
        also_speak_index_list.append(language)
        return also_speak_index_list.index(language)

In [10]:
def also_speak(teachers):
    also_speak_list = []
    for teacher in teachers:
        user_id = teacher['user_info']['user_id']
        for language in teacher['teacher_info']['also_speak']:
            current_language = language['language']
            index = check_language(current_language)
            id_language = {}
            id_language['user_id'] = user_id
            id_language['language'] = index
            # id_language['map'] = index
            also_speak_list.append(id_language)
    return also_speak_list

Record Book\
Takes a dictionary with the keys, min_price, max_price, number of records and datetime and append to a list

In [11]:
date_time_now = str(datetime.now(ZoneInfo('Asia/Singapore')))
date_time_now

'2026-06-02 11:05:05.031343+08:00'

In [12]:
def record_book(min_price, max_price, num_records, records_added):
    record_book_list = []
    date_time_now = str(datetime.now(ZoneInfo('Asia/Singapore')))
    records = {'min_price': min_price,
                'max_price': max_price,
                'total_records_called': num_records,
                'records_added': records_added,
                'date_time': date_time_now,
                'language': language}
    record_book_list.append(records)
    return record_book_list

Check number of entries per price range

In [150]:
min_price = 7100
max_price = min_price + 5999

In [151]:
# Change price here
def check_entries(min_price, max_price):
    dictionary = get_data(1, min_price, max_price)
    paging = dictionary['paging']
    return paging

page = check_entries(min_price, max_price)
print(page)
if int(page['total']) % 20 == 0:
    range_until = (int(page['total']) // 20) + 1
else:
    range_until = (int(page['total']) // 20) + 2
print(range_until)

{'page': 1, 'page_size': 20, 'total': 32, 'has_next': 1}
3


In [152]:
print(min_price, max_price, range_until, language)

7100 13099 3 french


Loop through pages

In [153]:
user_course_info_cycle = []
teacher_stats_cycle = []
pro_course_cycle = []
price_list_cycle = []
also_speak_list_cycle = []

has_next = 1
page = 1
print(min_price, max_price)
print('page 1')
while has_next == 1:
    api_call = get_data(page, min_price, max_price)
    teachers = api_call['data']
    user_course_info_cycle += user_course_info(teachers)
    teacher_stats_cycle += teacher_stats(teachers)
    price_list_buffer, pro_course_buffer = pro_course_prices(teachers)
    pro_course_cycle += pro_course_buffer
    price_list_cycle += price_list_buffer
    also_speak_list_cycle += also_speak(teachers)
    
    page += 1
    has_next = api_call['paging']['has_next']
    print('page', page)

records_added = join_data(user_course_info_cycle, 'user_course_info.json')
join_data(teacher_stats_cycle, 'teacher_stats.json')
join_data(pro_course_cycle, 'pro_course.json')
join_data(price_list_cycle, 'price_list.json')
join_data(also_speak_list_cycle, 'also_speaks.json')
override_data(also_speak_index_list, 'also_speaks_reference.json')

records_cycle = record_book(min_price, max_price, api_call['paging']['total'], records_added)

join_data(records_cycle, 'records.json')
# df = pd.DataFrame(user_course_info_list)


7100 13099
page 1
page 2
page 3


1

In [24]:
with open('also_speaks.json', 'r') as file:
    also_speak_list = json.load(file)

with open('also_speaks_reference.json', 'r') as file:
    also_speaks_index_list = json.load(file)

df_also_speak_list = pd.DataFrame(also_speak_list)
df_languages = pd.DataFrame(also_speak_index_list, columns=['other languages'])
result = pd.merge(df_also_speak_list, df_languages, left_on='language', right_index=True, how="left")
result

,user_id,language,other languages
0,5467830,0,spanish
1,5467830,1,arabic
2,5467830,2,arabic(maghrebi)
3,11401921,3,filipino(tagalog)
4,11401921,4,japanese
...,...,...,...
320,30589958,1,arabic
321,30589958,5,other
322,30408613,11,french
323,30408613,4,japanese


In [154]:
with open('user_course_info.json', 'r') as file:
    user_course_info_check = json.load(file)

df_user_course_info_check = pd.DataFrame(user_course_info_check)

In [155]:
df_user_course_info_check[df_user_course_info_check['user_id'].duplicated(keep=False)]

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
75,10654806,Kwabena,1,0,GH,GH,GH00001,Accra,GH00001,Accra,Atlantic/Reykjavik,2,0,550,600,116,0
76,28084220,Eileen,1,0,SG,JP,SG00000,Other,JP00000,Other,Asia/Tokyo,2,0,500,680,29,0
93,10599672,Naledi (Corporate) ✨,1,0,ZA,ZA,ZA00106,Klerksdorp,ZA00251,Sol Plaatje,Africa/Johannesburg,2,0,500,695,160,0
146,7522813,Mehdi El Quortobi,1,0,MA,MA,MA00036,Oujda,MA00036,Oujda,Africa/Casablanca,2,0,600,750,465,0
150,10009812,Ahmadi Nacer Eddine,1,0,DZ,DZ,DZ00146,El Oued,DZ00146,El Oued,Africa/Algiers,2,0,500,700,168,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5199,8418113,Lola French Teacher,1,0,FR,FR,FR00001,Paris,FR00016,Toulouse,Europe/Paris,2,0,3688,5500,266,1
5203,1671978,Vanessa,0,1,CA,CA,CA00006,Montreal,CA00006,Montreal,America/Toronto,2,0,1800,5500,0,1
5204,3204921,Rina E,1,1,CA,CA,CA00001,Toronto,CA00001,Toronto,America/Toronto,2,0,1500,5500,257,1
5226,1745587,Grace,1,0,TH,TH,TH00001,Bangkok,TH00001,Bangkok,Asia/Seoul,4,0,7000,8000,361,0


In [156]:
df_user_course_info_check['user_id'].nunique()

5076

In [157]:
df_user_course_info_check['user_id'].count()

np.int64(5234)

In [158]:
df_user_course_info_check

,user_id,nickname,is_tutor,is_pro,origin_country_id,living_country_id,origin_city_id,origin_city_name,living_city_id,living_city_name,timezone,trial_length,has_trial,trial_price,min_price,trial_session_count,has_beginner_course
0,5467830,⭐Learn with Kim/Kin,1,0,CN,KR,CN00001,Shanghai,KR00001,Seoul,Asia/Shanghai,2,0,1988,500,562,1
1,11401921,Teacher Emee,1,0,PH,PH,PH00000,Other,PH00000,Other,Asia/Manila,2,0,500,500,186,1
2,9627036,Shyam Syangtan,1,0,IN,IN,IN00771,Sonipat,IN00231,Delhi,Asia/Kolkata,2,0,700,500,1366,1
3,5787377,Kimi(y)a 👩🏻🎓❤️,1,0,IR,IT,IR00016,Tabriz,IT00016,Turin,Asia/Tehran,2,0,799,500,384,1
4,9934037,Anne Rola,1,0,PH,PH,PH00000,Other,PH00236,Tacloban,Asia/Shanghai,2,0,500,500,115,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5229,4877241,Yohann Coussot,0,1,FR,ES,FR00086,Dijon,ES00101,Santa Cruz de Tenerife,Atlantic/Canary,2,0,1500,7900,0,0
5230,12422499,🌈Lucille,0,1,FR,FR,FR00000,Other,FR00000,Other,Atlantic/Madeira,2,0,1200,7700,75,1
5231,6982686,Virginie,0,1,FR,PT,FR00051,Rennes,PT00001,Lisbon,Europe/Lisbon,2,0,3000,8000,0,0
5232,5067050,✨ Audrey ✨,0,1,FR,GB,FR00136,Metz,GB00251,Cambridge,Europe/London,2,0,500,8200,0,0


In [159]:
with open('pro_course.json', 'r') as file:
    pro_course_to_df = json.load(file)

df_pro_course = pd.DataFrame(pro_course_to_df)

In [160]:
df_pro_course[df_pro_course['language'] == 'french'].groupby('teacher_id').count()

,id,language,title,session_price,student_count,session_count,has_package
teacher_id,,,,,,,
222305,3,3,3,3,3,3,3
530071,6,6,6,6,6,6,6
614209,1,1,1,1,1,1,1
663702,5,5,5,5,5,5,5
740680,2,2,2,2,2,2,2
...,...,...,...,...,...,...,...
32117853,1,1,1,1,1,1,1
32117980,5,5,5,5,5,5,5
32164431,1,1,1,1,1,1,1


In [161]:
# df_user_course_info_check[df_user_course_info_check['nickname'] == 'Keegan Sparks']